In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

print("========== DATASET LOADING ==========")

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["Churn"]
)

X_train = train_df.drop(columns=["Churn", "customerID"])
y_train = train_df["Churn"].map({"No": 0, "Yes": 1})

X_test = test_df.drop(columns=["Churn", "customerID"])
y_test = test_df["Churn"].map({"No": 0, "Yes": 1})

print("Dataset Shape:", df.shape)
print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

print("\nTest data is reserved for final evaluation.")

========== DATASET LOADING ==========
Dataset Shape: (7043, 21)
Training Data: (5634, 19)
Testing Data: (1409, 19)

Test data is reserved for final evaluation.


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
import joblib

print("========== FINAL MODEL TRAINING ==========")

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = [
    col for col in X_train.columns
    if col not in numeric_features
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

scale_pos_weight = (
    (y_train == 0).sum() / (y_train == 1).sum()
)

final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    ))
])

final_model.fit(X_train, y_train)

print("Selected Model: XGBoost")
print("Final Model Trained Successfully")

========== FINAL MODEL TRAINING ==========
Selected Model: XGBoost
Final Model Trained Successfully


In [ ]:

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from xgboost import XGBClassifier

print("========== DATASET LOADING ==========")

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Dataset Shape:", df.shape)

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["Churn"]
)

X_train = train_df.drop(columns=["Churn", "customerID"])
y_train = train_df["Churn"].map({"No": 0, "Yes": 1})

X_test = test_df.drop(columns=["Churn", "customerID"])
y_test = test_df["Churn"].map({"No": 0, "Yes": 1})

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

print("\n========== PREPROCESSING ==========")

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [
    col for col in X_train.columns if col not in numeric_features
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print("\n========== FINAL MODEL TRAINING ==========")

final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    ))
])

final_model.fit(X_train, y_train)

print("Selected Model: XGBoost")
print("Final Model Trained Successfully")

print("\n========== TEST DATA PREDICTIONS ==========")

y_prob = final_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print("Predictions Generated:", len(y_pred))
print("Churn Probabilities Generated:", len(y_prob))

print("\n========== FINAL MODEL EVALUATION ==========")

roc_auc = roc_auc_score(y_test, y_prob)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("ROC-AUC Score:", round(roc_auc, 4))
print("F1 Score:", round(f1, 4))
print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["No Churn", "Churn"]
))

print("\n========== MODEL SAVING ==========")

joblib.dump(final_model, "final_churn_model.joblib")

print("Model saved: final_churn_model.joblib")

print("\n========== PREDICTIONS CSV ==========")

predictions_df = pd.DataFrame({
    "customerID": test_df["customerID"].values,
    "Actual_Churn": y_test.values,
    "Predicted_Churn": y_pred,
    "Churn_Probability": y_prob
})

predictions_df.to_csv("predictions.csv", index=False)

print("Predictions saved: predictions.csv")
print("Total Prediction Rows:", len(predictions_df))



========== DATASET LOADING ==========
Dataset Shape: (7043, 21)
Training Data: (5634, 19)
Testing Data: (1409, 19)

========== PREPROCESSING ==========

========== FINAL MODEL TRAINING ==========
Selected Model: XGBoost
Final Model Trained Successfully

========== TEST DATA PREDICTIONS ==========
Predictions Generated: 1409
Churn Probabilities Generated: 1409

========== FINAL MODEL EVALUATION ==========
ROC-AUC Score: 0.8426
F1 Score: 0.6346
Confusion Matrix:
[[770 265]
 [ 77 297]]

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.91      0.74      0.82      1035
       Churn       0.53      0.79      0.63       374

    accuracy                           0.76      1409
   macro avg       0.72      0.77      0.73      1409
weighted avg       0.81      0.76      0.77      1409


========== MODEL SAVING ==========
Model saved: final_churn_model.joblib

========== PREDICTIONS CSV ==========
Predictions saved: predictions.csv
Total Predict